In [ ]:
# 1. Install required packages
!pip install -q -U google-genai langchain-google-genai langchain-text-splitters chromadb sentence-transformers langgraph langchain-core

In [ ]:
import os
from google import genai
from google.colab import userdata

API_KEY = userdata.get("Gemini_API_Key")

if not API_KEY:
    raise ValueError("Add a valid Gemini API key to Colab Secrets as 'Gemini_API_Key'.")

client = genai.Client(api_key=API_KEY)

# Change this if your AI Studio account exposes a different Gemini model.
MODEL = "gemini-3.5-flash-lite"

print("Gemini client initialized.")


Gemini client initialized.


In [ ]:
import time
from google import genai
from google.colab import userdata

API_KEY = userdata.get("Gemini_API_Key")

if not API_KEY:
    raise ValueError("Add Gemini_API_Key to Colab Secrets.")

client = genai.Client(api_key=API_KEY)

MODEL = "gemini-3.5-flash-lite"

def ask_gemini(prompt, retries=3):
    for attempt in range(retries):
        try:
            result = client.models.generate_content(
                model=MODEL,
                contents=prompt
            )
            return result.text

        except Exception as e:
            if "503" in str(e):
                print(f"Gemini busy. Retrying {attempt + 1}/{retries}...")
                time.sleep(10)
            else:
                raise e

    raise RuntimeError(
        "Gemini is temporarily unavailable. Please try again later."
    )

print("Gemini client initialized.")

Gemini client initialized.


In [ ]:
lab_records = [
    {
        "id": "LAB001",
        "text": "PCR Experiment: A PCR reaction normally includes template DNA, primers, DNA polymerase, nucleotides and buffer. Typical stages are denaturation, annealing and extension. If amplification is absent, check template quality, primer design, reagent preparation and thermal cycling conditions."
    },
    {
        "id": "LAB002",
        "text": "Cell Culture Record: Cells should be maintained under suitable temperature, CO2 and sterile conditions. Contamination can appear as unexpected turbidity, rapid pH change or unusual cell morphology. Contaminated cultures should be isolated and handled according to laboratory biosafety procedures."
    },
    {
        "id": "LAB003",
        "text": "DNA Extraction Record: Low DNA yield can result from insufficient cell lysis, sample loss during purification, poor reagent quality or incorrect elution. DNA concentration and purity should be checked before downstream experiments."
    },
    {
        "id": "LAB004",
        "text": "Gel Electrophoresis Record: DNA bands can appear faint because of low DNA quantity, staining problems, sample loading errors or unsuitable electrophoresis conditions. A molecular ladder should be used for approximate size comparison."
    },
    {
        "id": "LAB005",
        "text": "Laboratory Safety Record: Use appropriate PPE, label samples clearly, maintain sterile technique where required, dispose of biological waste according to institutional rules, and report spills or contamination incidents to the responsible supervisor."
    }
]

print(f"Loaded {len(lab_records)} lab records.")


Loaded 5 lab records.


In [ ]:
print(ask_gemini(
    "Explain DNA replication in 3 simple points."
))

Gemini busy. Retrying 1/3...
Here is DNA replication in 3 simple points:

1. **Unzipping:** An enzyme (helicase) unwinds and "unzips" the double-stranded DNA molecule, breaking it apart down the middle like a zipper. 
2. **Matching:** Free-floating building blocks (nucleotides) in the cell match up with the exposed bases on each of the single strands following strict pairing rules (A with T, and C with G).
3. **Sealing:** Enzymes (like DNA polymerase) glue the new pieces together, resulting in **two identical DNA molecules**, each containing one old strand and one new strand.


In [ ]:
def retrieve_lab_records(query, top_k=3):
    query_embedding = embedder.encode([query]).tolist()

    result = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )

    docs = result["documents"][0]
    ids = result["ids"][0]

    return list(zip(ids, docs))

# Test RAG
question = "Why might a PCR experiment have no DNA amplification?"
results = retrieve_lab_records(question)

for record_id, text in results:
    print(f"\n[{record_id}]\n{text}")



[LAB001]
PCR Experiment: A PCR reaction normally includes template DNA, primers, DNA polymerase, nucleotides and buffer. Typical stages are denaturation, annealing and extension. If amplification is absent, check template quality, primer design, reagent preparation and thermal cycling conditions.

[LAB003]
DNA Extraction Record: Low DNA yield can result from insufficient cell lysis, sample loss during purification, poor reagent quality or incorrect elution. DNA concentration and purity should be checked before downstream experiments.

[LAB004]
Gel Electrophoresis Record: DNA bands can appear faint because of low DNA quantity, staining problems, sample loading errors or unsuitable electrophoresis conditions. A molecular ladder should be used for approximate size comparison.


In [ ]:
from langchain_core.tools import StructuredTool

def search_lab_records(query: str) -> str:
    """Search the biotechnology laboratory knowledge base for relevant records."""
    results = retrieve_lab_records(query, top_k=3)

    if not results:
        return "No relevant lab records found."

    return "\n\n".join(
        f"[{record_id}] {text}"
        for record_id, text in results
    )

lab_search_tool = StructuredTool.from_function(
    func=search_lab_records,
    name="search_lab_records",
    description="Searches biotechnology lab records for protocols, observations, problems and safety information."
)

print("Lab search tool created.")


Lab search tool created.


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class LabState(TypedDict):
    user_query: str
    retrieved_context: str
    analysis: str
    response: str
    is_valid: bool


def retrieve_node(state: LabState):
    context = search_lab_records(state["user_query"])
    return {"retrieved_context": context}


def analyze_node(state: LabState):
    prompt = f"""
You are a biotechnology laboratory supervisor.

User question:
{state["user_query"]}

Retrieved laboratory information:
{state["retrieved_context"]}

Give a short analysis:
1. What is likely happening?
2. What should be checked?
3. What safety point matters?

Do not invent experimental results.
"""

    analysis = ask_gemini(prompt)

    return {"analysis": analysis}


def generate_node(state: LabState):
    prompt = f"""
You are a Biotechnology Lab Supervisor AI.

User question:
{state["user_query"]}

Retrieved laboratory information:
{state["retrieved_context"]}

Analysis:
{state["analysis"]}

Give a concise answer with:

- Observation
- Possible cause
- Recommended checks
- Safety note

Use simple language.
"""

    response = ask_gemini(prompt)

    return {"response": response}


def validate_node(state: LabState):
    text = state["response"].strip()

    valid = len(text) > 30 and "Observation" in text

    return {"is_valid": valid}


def router(state: LabState):
    if state["is_valid"]:
        return "approved"
    else:
        return "retry"


workflow = StateGraph(LabState)

workflow.add_node("retrieve", retrieve_node)
workflow.add_node("analyze", analyze_node)
workflow.add_node("generate", generate_node)
workflow.add_node("validate", validate_node)

workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "analyze")
workflow.add_edge("analyze", "generate")
workflow.add_edge("generate", "validate")

workflow.add_conditional_edges(
    "validate",
    router,
    {
        "approved": END,
        "retry": "generate"
    }
)

app = workflow.compile()

print("LangGraph workflow compiled successfully.")

LangGraph workflow compiled successfully.


## 8. Run the Complete AI Supervisor

Try a biotechnology laboratory question.


In [ ]:
def run_lab_supervisor(user_query):
    result = app.invoke({
        "user_query": user_query,
        "retrieved_context": "",
        "analysis": "",
        "response": "",
        "is_valid": False
    })

    print("\n========== BIOTECHNOLOGY LAB SUPERVISOR AI ==========")
    print("\nQuestion:")
    print(user_query)

    print("\nRetrieved Records:")
    print(result["retrieved_context"])

    print("\nFinal Supervisor Response:")
    print(result["response"])

## 9. Interactive Question

Enter your own laboratory question.


In [32]:
user_question = input("Enter your lab question: ")
run_lab_supervisor(user_question)

Enter your lab question: Explain DNA replication in 3 simple points.

========== BIOTECHNOLOGY LAB SUPERVISOR AI ==========

Question:
Explain DNA replication in 3 simple points.

Retrieved Records:
[LAB001] PCR Experiment: A PCR reaction normally includes template DNA, primers, DNA polymerase, nucleotides and buffer. Typical stages are denaturation, annealing and extension. If amplification is absent, check template quality, primer design, reagent preparation and thermal cycling conditions.

[LAB003] DNA Extraction Record: Low DNA yield can result from insufficient cell lysis, sample loss during purification, poor reagent quality or incorrect elution. DNA concentration and purity should be checked before downstream experiments.

[LAB004] Gel Electrophoresis Record: DNA bands can appear faint because of low DNA quantity, staining problems, sample loading errors or unsuitable electrophoresis conditions. A molecular ladder should be used for approximate size comparison.

Final Supervisor